In [ ]:
import os
import sys
import warnings

# Garante que o Python encontra a pasta 'src'
sys.path.insert(0, os.path.abspath('..'))
from src import data
from src import experiment as E
from src import metrics as M
import numpy as np

print("Loading and preparing data...")
d = E.setup(seed=42)
tr, va = d['tr'], d['va']

# Usar as matrizes densas desduplicadas
Xa_dense, Ya, cnt = E.dedup_configs(d['X_train'][tr], d['Y_cons'][tr])
X_val_dense = d['X_train'][va] 
Yv = d['Y_cons'][va]

print(f"✅ Data ready! Training matrix shape (Dense): {Xa_dense.shape}")

In [ ]:
from catboost import CatBoostClassifier

print("Initializing CatBoost Multi-Output Classifier...")

# O modelo único que aprende as dependências conjuntas nativamente
cb_model = CatBoostClassifier(
    iterations=500,                # Quantidade de árvores
    learning_rate=0.1,             # Tamanho do passo
    depth=6,                       # Profundidade de cada árvore
    loss_function='MultiCrossEntropy', # A função chave para Multi-Label
    custom_metric=['HammingLoss'], # Métrica acompanhada no treino
    random_seed=42,
    verbose=100                    # Imprime progresso a cada 100 árvores
)

with E.Timer() as t:
    print("Training a single multi-output forest... 🚀")
    cb_model.fit(Xa_dense, Ya)
    
print(f"✅ Training completed in {t.s:.0f} seconds.")

In [ ]:
from IPython.display import display

print("Predicting probabilities on validation set...")
# O CatBoost retorna um array (linhas x colunas) tal como esperamos
P_val = cb_model.predict_proba(X_val_dense)

# Binarizar a probabilidade usando o limiar clássico de 0.5
pred_val = (P_val >= 0.5).astype(np.uint8)

print("Applying Sprint 1 additive rules...")
pred_val_final = data.apply_additive_rules(
    pred_val, 
    d['X_train'][va], 
    d['crm_cols'], 
    d['bil_cols']
)

print("\n🎯 RESULTS: CATBOOST MULTI-OUTPUT + RULES 🎯")
val_summary, val_cm, val_f1 = M.multilabel_report(Yv, pred_val_final, P_val)
display(val_summary.to_frame('CatBoost Validation').style.format('{:.5f}'))

In [ ]:
import pandas as pd

# Structure the model record
catboost_result = {
    'exp_id': 'E-TierC-CatBoost-Multi',
    'pipeline': 'CatBoost MultiCrossEntropy + Additive Rules',
    'family': 'CatBoostClassifier',
    'params': 'iterations=500, depth=6, loss=MultiCrossEntropy',
    'emr': val_summary['EMR'],
    'hamming_loss': val_summary['Hamming loss'],
    'train_time_s': t.s, # Assuming the timer variable 't' is still in memory
    'notes': 'EMR 83.04%. Median F1 collapsed to 0! Proves that global MultiCrossEntropy squashes rare labels in favor of Hamming Loss, validating the use of Binary Relevance in the Champion model.'
}

# Load, update, and save the tracker
tracker_path = '../data/derived/sprint2_experiment_results.csv'
results_df = pd.read_csv(tracker_path)
results_df = results_df[results_df['exp_id'] != 'E-TierC-CatBoost-Multi'] # Avoid duplicates
results_df = pd.concat([results_df, pd.DataFrame([catboost_result])], ignore_index=True)
results_df.to_csv(tracker_path, index=False)

print("✅ CatBoost Multi-Output model successfully added to the team tracker!")